Realizamos las importaciones necesarias:

In [198]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.base import clone
from joblib import parallel_backend
import torch
from transformers import AutoModel, AutoTokenizer

Fijamos las variables globales del cuaderno:

In [103]:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "etiquetas_test_sin_alias.csv")
EMB_FILE = "embeddings_specter2.npz"
SPECTER2 = "allenai/specter2_base"
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

Definimos funciones de carga de datos, tokenización, y cálculo de métricas y estadísticas:

In [282]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=','):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Creamos un set para los títulos ya encontrados a fin de evitar duplicados:
        titles = set()

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            title = row[0].strip().lower()
            summary = row[1].strip().lower()

            if title not in titles:
                titles.add(title)
            else:
                continue

            papers.append((title, summary))
            terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    probs = {}
    for label, freq in sorted(ct.items(), key=lambda item: item[1], reverse=True):
        prob = freq / len(texts)
        probs[label] = prob
        print(f"{label}: {freq} ({prob:.1%})", end="  ")
    print("\n")
    return probs


try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

## Specter2

### Entrenamiento de una SVM con *embeddings*

Cargamos el modelo (Specter2) y su tokenizador, y fijamos el dispositivo de ejecución:

In [105]:
# Cargamos el modelo y el tokenizador de Specter2:
tokenizer_s2 = AutoTokenizer.from_pretrained(SPECTER2)
model_s2 = AutoModel.from_pretrained(SPECTER2)

# Fijamos el dispositivo de ejecución y establecemos el modelo en modo evaluación:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"
print("Dispositivo utilizado:", device)
model_s2.to(device)
model_s2.eval();

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2716.33it/s]


Dispositivo utilizado: cuda


Definimos una función para extraer los embeddings de nuestros papers con el modelo congelado:

In [106]:
def get_embeddings(papers, model, tokenizer, batch_size=32):
    embeddings = []

    # Como estamos en inferencia, desactivamos el cálculo de gradientes:
    with torch.no_grad():
        for i in range(0, len(papers), batch_size):

            # Confeccionamos un batch de 32 papers:
            batch_papers = papers[i : i + batch_size]

            # Tokenizamos cada paper:
            inputs = tokenizer(
                batch_papers,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)

            # Recibimos los embeddings asociados a los papers tokenizados (reducimos los parámetros a 16 bits):
            with torch.autocast(
                device_type="cuda" if is_cuda else "cpu",
                dtype=torch.float16 if is_cuda else torch.bfloat16,
                enabled=is_cuda
            ):
                outputs = model(**inputs)

            # Tomamos el embedding del token [CLS] (índice 0) para todo el batch:
            cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_vectors)

    return np.vstack(embeddings)

Cargamos el dataset de entrenamiento y reservamos una partición de validación:

In [107]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
train_tok, dev_tok, y_train, y_dev = train_test_split(all_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

Papers de entrenamiento: 32570
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


Cargamos el conjunto de test: [BORRAR LA CARGA DE LAS ETIQUETAS AL ENTREGAR]

In [108]:
raw_test, y_test_nobin = load_corpus(TEST_FILE, ";")
print("Papers de test:", len(raw_test))

y_test = mlb.transform(y_test_nobin)

test_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_test]

Papers de test: 4885


Obtenemos los embeddings de los papers de entrenamiento y de test:

In [ ]:
# Si ya teníamos un archivo de embeddings guardado, lo cargamos:
if os.path.exists(EMB_FILE):
    data = np.load(EMB_FILE)
    X_all_emb, X_test_emb = data["all"], data["test"]

# Si no, los calculamos de cero:
else:
    X_all_emb = get_embeddings(all_tok, model_s2, tokenizer_s2, batch_size=32)
    X_test_emb = get_embeddings(test_tok, model_s2, tokenizer_s2, batch_size=32)

    # Guardamos los embeddings en un archivo comprimido para no tener que recalcularlos al reiniciar el kernel:
    np.savez_compressed(EMB_FILE, all=X_all_emb, test=X_test_emb)

# Normalizamos los embeddings para favorecer la convergencia de SVM:
X_all_emb_norm = normalize(X_all_emb, norm="l2")
X_test_emb_norm = normalize(X_test_emb, norm="l2")

Extraemos el conjunto de validación de los embeddings de entrenamiento:

In [111]:
X_train_emb_norm, X_dev_emb_norm = train_test_split(X_all_emb_norm, test_size=DEV_PART, shuffle=True, random_state=SEED)

Definimos funciones para ajustar los umbrales de decisión y realizar predicciones con dichos umbrales ajustados:

In [112]:
# Definimos una función para ajustar los umbrales de decisión de un clasificador:
def adjust_thresholds(model, X, y, min_th=0.01, max_th=1, n_th=200, alg="rl"):
    # Obtenemos las probabilidades para cada clase que ha asociado el modelo a cada paper:
    scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Creamos un vector con tantos elementos como clases, inicializado a ceros:
    n_classes = len(mlb.classes_)
    optimal_thresholds = np.zeros(n_classes)

    # Definimos una lista de 'n_th' umbrales de decisión equiespaciados entre 'min_th' y 'max_th':
    thresholds = np.linspace(min_th, max_th, n_th)

    # Iteramos sobre cada clase:
    for c in range(n_classes):
        max_f1_c = -1
        best_th = 0

        # Obtenemos la columna de etiquetas y probabilidades para cada paper asociadas a la clase:
        y_c = y[:, c]
        scores_c = scores[:, c]

        # Iteramos sobre cada umbral:
        for th in thresholds:
            # Obtenemos las predicciones de etiquetas asociadas a ese umbral y calculamos su F1 individual:
            preds_c = (scores_c >= th).astype(int)
            f1_c = f1_score(y_c, preds_c, average="binary", zero_division=0)

            # Si el F1 mejora respecto al que tenía la clase ya, lo guardamos:
            if f1_c > max_f1_c:
                max_f1_c = f1_c
                best_th = th

        # En cada entrada 'i' de 'optimal_thresholds', almacenamos el mejor umbral encontrado para la clase 'c_i':
        optimal_thresholds[c] = best_th

    return optimal_thresholds

In [113]:
# Definimos asimismo una función para obtener las predicciones optimizadas dados los umbrales ajustados:
def predict_with_th(model, X, y, thresholds, verbose=False, alg="rl"):
    # Obtenemos las probabilidades originales:
    scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Actualizamos las predicciones con los umbrales de decisión que hemos calculado:
    preds = (scores >= thresholds).astype(int)

    # Buscamos los papers que se hayan quedado sin clase:
    zero_rows = np.where(preds.sum(axis=1) == 0)[0] # NumPy devuelve una tupla de 1 elemento, por eso lo extraemos

    # Iteramos sobre cada paper sin clase, obtenemos la clase de mayor probabilidad y se la asignamos:
    for r in zero_rows:
        top_class = np.argmax(scores[r])
        preds[r, top_class] = 1

    if verbose:
        # Consultamos las métricas derivadas de usarlos:
        print("=== Métricas con umbrales de decisión optimizados ===")
        print(compute_metrics(y, preds))

    return preds

Definimos un SVM a entrenar con los embeddings normalizamos y ajustamos su valor C:

In [114]:
svm_s2 = {}
best_c_s2, best_f1_s2 = None, -1
for c in [0.5, 1, 3, 5, 7, 10]:
    # Definimos nuestro SVM a entrenar:
    svm_s2[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual="auto",
            class_weight="balanced",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=8,
    ).fit(X_train_emb_norm, y_train)

    print(f"== C={c}:")
    metrics = compute_metrics(y_dev, svm_s2[c].predict(X_dev_emb_norm))
    print(metrics)
    print()

    # Si la F1 macro del modelo actual es la mejor hasta el momento, nos guardamos su C asociada:
    if metrics["f1"] > best_f1_s2:
        best_f1_s2 = metrics["f1"]
        best_c_s2 = c

== C=0.5:
{'accuracy': 0.10417519443307409, 'f1': 0.23974944701666015, 'precision': 0.17865560751620865, 'recall': 0.7351894394755102, 'hamming_loss': 0.07256885562247822}

== C=1:
{'accuracy': 0.11440851412198116, 'f1': 0.25090235907984765, 'precision': 0.18630147163893226, 'recall': 0.7106351907985422, 'hamming_loss': 0.06828548038126425}

== C=3:
{'accuracy': 0.14613180515759314, 'f1': 0.27218789453389386, 'precision': 0.20231565045859354, 'recall': 0.6595465855558066, 'hamming_loss': 0.06153148938658558}

== C=5:
{'accuracy': 0.15943512075317232, 'f1': 0.27897974016029525, 'precision': 0.2080801240997653, 'recall': 0.6348897020270943, 'hamming_loss': 0.05890006432372376}

== C=7:
{'accuracy': 0.16946377404830126, 'f1': 0.2862387424387212, 'precision': 0.21380822363360766, 'recall': 0.6194163355069925, 'hamming_loss': 0.05740892345476873}

== C=10:
{'accuracy': 0.1788784281620958, 'f1': 0.2868045246557376, 'precision': 0.21668731902454447, 'recall': 0.5926648217977856, 'hamming_loss

In [118]:
for c in [6, 8, 9, 12]:
    # Definimos nuestro SVM a entrenar:
    svm_s2[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual="auto",
            class_weight="balanced",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=8,
    ).fit(X_train_emb_norm, y_train)

    print(f"== C={c}:")
    metrics = compute_metrics(y_dev, svm_s2[c].predict(X_dev_emb_norm))
    print(metrics)
    print()

    # Si la F1 macro del modelo actual es la mejor hasta el momento, nos guardamos su C asociada:
    if metrics["f1"] > best_f1_s2:
        best_f1_s2 = metrics["f1"]
        best_c_s2 = c

== C=6:
{'accuracy': 0.16311911584117889, 'f1': 0.2823347347870298, 'precision': 0.21068970778492688, 'recall': 0.6278064958504712, 'hamming_loss': 0.05824951757207181}

== C=8:
{'accuracy': 0.17151043798608268, 'f1': 0.284691495894444, 'precision': 0.21355909002374923, 'recall': 0.605893648361362, 'hamming_loss': 0.056911876498450385}

== C=9:
{'accuracy': 0.17560376586164553, 'f1': 0.28689772545160536, 'precision': 0.21591079783955708, 'recall': 0.6010790705587411, 'hamming_loss': 0.05637828197181451}

== C=12:
{'accuracy': 0.1829717560376586, 'f1': 0.29565643161326677, 'precision': 0.22431795054602602, 'recall': 0.5911896175314938, 'hamming_loss': 0.05506987895444711}



Para el mejor valor de C que hayamos encontrado, ajustamos los umbrales de decisión del modelo y realizamos una predicción:

In [213]:
char_th = adjust_thresholds(svm_s2[best_c_s2], X_dev_emb_norm, y_dev, -2, 2, 500, alg="svm")
preds_test = predict_with_th(svm_s2[best_c_s2], X_test_emb_norm, y_test, char_th, verbose=True, alg="svm")

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.23418628454452406, 'f1': 0.19027822299108657, 'precision': 0.1596128789143388, 'recall': 0.3085381589981303, 'hamming_loss': 0.04665886825559292}


### Inferencia *zero-shot* mediante similitud coseno

Empezaremos confeccionando un diccionario con las descripciones de las 28 etiquetas que aparecen en el dataset de entrenamiento, obtenidas directamente de la taxonomía de arXiv:

In [121]:
desc_arxiv = {
    "cs.AI": "Artificial Intelligence: Covers all areas of AI except Vision, Robotics, Machine Learning, Multiagent Systems, and Computation and Language (Natural Language Processing), which have separate subject areas. In particular, includes Expert Systems, Theorem Proving (although this may overlap with Logic in Computer Science), Knowledge Representation, Planning, and Uncertainty in AI. Roughly includes material in General Artificial Intelligence, Applications and Expert Systems, Deduction and Theorem Proving, Knowledge Representation Formalisms and Methods, Problem Solving, Control Methods, and Search, and Distributed Artificial Intelligence.",
    "cs.CL": "Computation and Language: Covers natural language processing. Roughly includes material in Natural Language Processing. Note that work on artificial languages (programming languages, logics, formal systems) that does not explicitly address natural-language issues broadly construed (natural-language processing, computational linguistics, speech, text retrieval, etc.) is not appropriate for this area.",
    "cs.CR": "Cryptography and Security: Covers all areas of cryptography and security including authentication, public key cryptosytems, proof-carrying code, etc. Roughly includes material in Security and Protection, and Data Encryption.",
    "cs.CV": "Computer Vision and Pattern Recognition: Covers image processing, computer vision, pattern recognition, and scene understanding. Roughly includes material in Vision and Scene Understanding, Image Processing and Computer Vision, and Pattern Recognition.",
    "cs.CY": "Computers and Society: Covers impact of computers on society, computer ethics, information technology and public policy, legal aspects of computing, computers and education. Roughly includes material in General Computing Milieux, History of Computing, Computers and Education, Computers and Society, Legal Aspects of Computing, and The Computing Profession.",
    "cs.DC": "Distributed, Parallel, and Cluster Computing: Covers fault-tolerance, distributed algorithms, stabilility, parallel computation, and cluster computing. Roughly includes material in Multiple Data Stream Architectures (Multiprocessors), Parallel Architectures, Distributed Systems, Concurrent Programming, Operating Systems Reliability, Operating Systems Organization and Design, and Data Structures.",
    "cs.GR": "Graphics: Covers all aspects of computer graphics. Roughly includes material in all of Computer Graphics, except that Computational Geometry and Object Modeling is likely to have Computational Geometry as the primary subject area.",
    "cs.HC": "Human-Computer Interaction: Covers human factors, user interfaces, and collaborative computing. Roughly includes material in User/Machine Systems and all of Information Interfaces and Presentation (e.g., HCI), except for Multimedia Information Systems, which is more likely to have Multimedia as the primary subject area.",
    "cs.IR": "Information Retrieval: Covers indexing, dictionaries, retrieval, content and analysis. Roughly includes material in General Information Storage and Retrieval, Content Analysis and Indexing, Information Storage, Information Search and Retrieval, and Systems and Software.",
    "cs.IT": "Information Theory: Covers theoretical and experimental aspects of information theory and coding. Includes material in Coding and Information Theory and intersects with Systems and Information Theory.",
    "cs.LG": "Machine Learning: Papers on all aspects of machine learning research (supervised, unsupervised, reinforcement learning, bandit problems, and so on) including also robustness, explanation, fairness, and methodology. cs.LG is also an appropriate primary category for applications of machine learning methods.",
    "cs.MA": "Multiagent Systems: Covers multiagent systems, distributed artificial intelligence, intelligent agents, coordinated interactions. and practical applications. Roughly covers Distributed Artificial Intelligence.",
    "cs.MM": "Multimedia: Roughly includes material in Multimedia Information Systems.",
    "cs.NE": "Neural and Evolutionary Computing: Covers neural networks, connectionism, genetic algorithms, artificial life, adaptive behavior. Roughly includes some material in Other Architecture Styles, Learning, and Pattern Recognition.",
    "cs.RO": "Robotics: Roughly includes material in Robotics.",
    "cs.SI": "Social and Information Networks: Covers the design, analysis, and modeling of social and information networks, including their applications for on-line information access, communication, and interaction, and their roles as datasets in the exploration of questions in these and other domains, including connections to the social and biological sciences. Analysis and modeling of such networks includes topics in Analysis of Algorithms and Problem Complexity, Discrete Mathematics, Probability and Statistics, Database Management, and Artificial Intelligence; applications in computing include topics in Information Storage and Retrieval, Information Systems Applications, and Information Interfaces and Presentation; and applications at the interface of computing and other disciplines include topics in Administrative Data Processing, Physical Sciences and Engineering, Life and Medical Sciences, Social and Behavioral Sciences, Arts and Humanities, Computer-Aided Engineering, and Computers in Other Systems. Papers on computer communication systems and network protocols (e.g. TCP/IP) are generally a closer fit to the Networking and Internet Architecture (cs.NI) category.",
    "cs.SY": "Systems and Control: cs.SY is an alias for eess.SY. This section includes theoretical and experimental research covering all facets of automatic control systems. The section is focused on methods of control system analysis and design using tools of modeling, simulation and optimization. Specific areas of research include nonlinear, distributed, adaptive, stochastic and robust control in addition to hybrid and discrete event systems. Application areas include automotive and aerospace control systems, network control, biological systems, multiagent and cooperative control, robotics, reinforcement learning, sensor networks, control of cyber-physical and energy-related systems, and control of computing systems.",
    "eess.IV": "Image and Video Processing: Theory, algorithms, and architectures for the formation, capture, processing, communication, analysis, and display of images, video, and multidimensional signals in a wide variety of applications. Topics of interest include: mathematical, statistical, and perceptual image and video modeling and representation; linear and nonlinear filtering, de-blurring, enhancement, restoration, and reconstruction from degraded, low-resolution or tomographic data; lossless and lossy compression and coding; segmentation, alignment, and recognition; image rendering, visualization, and printing; computational imaging, including ultrasound, tomographic and magnetic resonance imaging; and image and video analysis, synthesis, storage, search and retrieval.",
    "eess.SP": "Signal Processing: Theory, algorithms, performance analysis and applications of signal and data analysis, including physical modeling, processing, detection and parameter estimation, learning, mining, retrieval, and information extraction. The term \"signal\" includes speech, audio, sonar, radar, geophysical, physiological, (bio-) medical, image, video, and multimodal natural and man-made signals, including communication signals and data. Topics of interest include: statistical signal processing, spectral estimation and system identification; filter design, adaptive filtering / stochastic learning; (compressive) sampling, sensing, and transform-domain methods including fast algorithms; signal processing for machine learning and machine learning for signal processing applications; in-network and graph signal processing; convex and nonconvex optimization methods for signal processing applications; radar, sonar, and sensor array beamforming and direction finding; communications signal processing; low power, multi-core and system-on-chip signal processing; sensing, communication, analysis and optimization for cyber-physical systems such as power grids and the Internet of Things.",
    "eess.SY": "Systems and Control: This section includes theoretical and experimental research covering all facets of automatic control systems. The section is focused on methods of control system analysis and design using tools of modeling, simulation and optimization. Specific areas of research include nonlinear, distributed, adaptive, stochastic and robust control in addition to hybrid and discrete event systems. Application areas include automotive and aerospace control systems, network control, biological systems, multiagent and cooperative control, robotics, reinforcement learning, sensor networks, control of cyber-physical and energy-related systems, and control of computing systems.",
    "math.IT": "Information Theory: math.IT is an alias for cs.IT. Covers theoretical and experimental aspects of information theory and coding.",
    "math.OC": "Optimization and Control: Operations research, linear programming, control theory, systems theory, optimal control, game theory",
    "math.ST": "Statistics Theory: Applied, computational and theoretical statistics: e.g. statistical inference, regression, time series, multivariate analysis, data analysis, Markov chain Monte Carlo, design of experiments, case studies",
    "physics.chem-ph": "Chemical Physics: Experimental, computational, and theoretical physics of atoms, molecules, and clusters - Classical and quantum description of states, processes, and dynamics; spectroscopy, electronic structure, conformations, reactions, interactions, and phases. Chemical thermodynamics. Disperse systems. High pressure chemistry. Solid state chemistry. Surface and interface chemistry.",
    "stat.AP": "Applications: Biology, Education, Epidemiology, Engineering, Environmental Sciences, Medical, Physical Sciences, Quality Control, Social Sciences",
    "stat.ME": "Methodology: Design, Surveys, Model Selection, Multiple Testing, Multivariate Methods, Signal and Image Processing, Time Series, Smoothing, Spatial Statistics, Survival Analysis, Nonparametric and Semiparametric Methods",
    "stat.ML": "Machine Learning: Covers machine learning papers (supervised, unsupervised, semi-supervised learning, graphical models, reinforcement learning, bandits, high dimensional inference, etc.) with a statistical or theoretical grounding",
    "stat.TH": "Statistics Theory: stat.TH is an alias for math.ST. Asymptotics, Bayesian Inference, Decision Theory, Estimation, Foundations, Inference, Testing."
}

# Confeccionamos una lista de las descripciones siguiendo el orden de 'mlb.classes_':
desc_list = [desc_arxiv[c] for c in mlb.classes_]

Obtenemos los embeddings de las descripciones y los normalizamos:

In [125]:
# Si ya los teníamos guardados, los cargamos:
with np.load(EMB_FILE) as data:
    if "tags" in data:
        desc_emb = data["tags"]

    # Si no, los calculamos:
    else:
        desc_emb = get_embeddings(desc_list, model_s2, tokenizer_s2, batch_size=32)
        dict_data = dict(data)
        dict_data["tags"] = desc_emb
        np.savez_compressed(EMB_FILE, **dict_data)

# Normalizamos los embeddings:
desc_emb_norm = normalize(desc_emb, norm="l2")

Obtenemos la matriz de similitud semática entre los embeddings de test y los de las descripciones:

In [216]:
# Multiplicación matricial: (N_test, 768) @ (768, 28) -> (N_test, 28)
sim_all = np.dot(X_all_emb_norm, desc_emb_norm.T)
sim_test = np.dot(X_test_emb_norm, desc_emb_norm.T)

In [218]:
sim_test[0]

array([0.8779776 , 0.8417316 , 0.77984375, 0.8180829 , 0.7744748 ,
       0.81917036, 0.81021357, 0.81213665, 0.84143084, 0.8283744 ,
       0.8857281 , 0.83909345, 0.80666345, 0.8442924 , 0.85790825,
       0.8100934 , 0.8368279 , 0.78790545, 0.8002147 , 0.8325793 ,
       0.816241  , 0.81966317, 0.8157122 , 0.76612055, 0.79709864,
       0.7922898 , 0.89307797, 0.8305378 ], dtype=float32)

Como todos los valores se encuentran en el rango [0.79, 0.90], vamos a **estandarizarlos** (restarles la media y dividirlos entre la desviación típica) para estudiar cuáles destacan especialmente:

In [220]:
# Sumamos un pequeño epsilon a la desviación típica para evitar divisiones por cero:
sim_all_std = (sim_all - sim_all.mean(axis=0)) / (sim_all.std(axis=0) + 1e-8)

In [221]:
sim_all_std[0]

array([-0.7925652 , -1.4550517 , -1.036779  ,  1.0823911 , -0.79426205,
       -1.2371016 ,  0.9411497 , -0.7259197 , -1.6158211 , -1.4632534 ,
       -1.6521306 , -1.3555741 , -0.48787388, -1.015032  ,  0.06451676,
       -1.3844386 , -1.3679028 ,  0.48708028, -0.778298  , -1.3933032 ,
       -1.1246204 , -1.2911131 , -1.3808861 , -1.3118374 , -1.3520818 ,
       -0.73587316, -2.090055  , -1.1206576 ], dtype=float32)

In [247]:
# Z-scores por documento en train: (N_train, 28)
z_train = sim_all_std

# Densidad media real de etiquetas en train (ej. 1.4)
etiquetas_medias_train = y_bin.sum(axis=1).mean() - 0.4

# Buscamos el umbral z_th en train tal que la media de etiquetas predichas coincida
# Probamos valores de z entre 1.0 y 2.5
z_candidatos = np.linspace(1.0, 2.5, 100)
pred_medias = np.array([(z_train >= z).sum(axis=1).mean() for z in z_candidatos])

# El Z óptimo es aquel donde la densidad predicha iguala a la densidad real
idx_optimo = np.argmin(np.abs(pred_medias - etiquetas_medias_train))
z_th_optimo = z_candidatos[idx_optimo]

# Ahora aplicas z_th_optimo en test:
z_test = (sim_test - sim_test.mean(axis=1, keepdims=True)) / (sim_test.std(axis=1, keepdims=True) + 1e-8)
preds_test = (z_test >= 1.3).astype(int)

In [273]:
print(compute_metrics(y_test, y_test))

{'accuracy': 1.0, 'f1': 0.8571428571428571, 'precision': 0.8571428571428571, 'recall': 0.8571428571428571, 'hamming_loss': 0.0}


In [276]:
modelo_optimo = svm_s2[best_c_s2]
preds_dev = modelo_optimo.predict(X_dev_emb_norm)
preds_test = modelo_optimo.predict(X_test_emb_norm)

In [277]:
import numpy as np

# 1. Obtener las predicciones binarias normales (0 o 1) de los SVMs
# (Asumiendo que ya tienes preds_dev y preds_test)
# preds_dev.shape = (N_dev, 28), y_dev.shape = (N_dev, 28)
# preds_test.shape = (N_test, 28)

p_test_estimada = np.zeros(28)

for c in range(28):
    # --- FASE 1: Calibrar la caja negra en DEV ---
    # Positivos y negativos reales de la clase c en validación
    reales_pos = (y_dev[:, c] == 1)
    reales_neg = (y_dev[:, c] == 0)
    
    # True Positive Rate (Recall): de los reales positivos, ¿cuántos predijo el SVM como 1?
    if reales_pos.sum() > 0:
        tpr = preds_dev[reales_pos, c].mean()
    else:
        tpr = 1.0  # Fallback si no hay positivos en dev
        
    # False Positive Rate: de los reales negativos, ¿cuántos predijo el SVM como 1?
    if reales_neg.sum() > 0:
        fpr = preds_dev[reales_neg, c].mean()
    else:
        fpr = 0.0

    # --- FASE 2: Invertir sobre las predicciones de TEST ---
    # q es la proporción de veces que el modelo HA PREDICHO la clase c en test
    q_test = preds_test[:, c].mean()
    
    # Denominador de la matriz invertida
    denominador = tpr - fpr
    
    # Prevenir divisiones por cero si el clasificador predice al azar (TPR == FPR)
    if abs(denominador) < 1e-4:
        p_test_estimada[c] = q_test
    else:
        # Fórmula BBSE binaria
        p_test_estimada[c] = (q_test - fpr) / denominador

# 3. Recortar desviaciones numéricas para mantener probabilidades válidas
p_test_estimada = np.clip(p_test_estimada, 0.0, 1.0)

In [278]:
p_test_estimada

array([0.58607274, 0.24653417, 0.01195352, 0.57782651, 0.01307112,
       0.00315079, 0.04827752, 0.00628688, 0.        , 0.        ,
       0.58447662, 0.00708014, 0.02505233, 0.        , 0.04744767,
       0.        , 0.01077067, 0.10427666, 0.0452571 , 0.01077067,
       0.        , 0.        , 0.        , 0.00409668, 0.00938143,
       0.00081883, 0.2453984 , 0.        ])

In [279]:
scores_test = modelo_optimo.decision_function(X_test_emb_norm)
char_th_bbse = np.zeros(28)

for c in range(28):
    p_c = p_test_estimada[c] #
    if p_c <= 1e-5:
        # Si la prevalencia estimada es prácticamente 0, no predecir nunca
        char_th_bbse[c] = np.inf
    else:
        # El corte se sitúa en el percentil (1 - p_c) de los scores de test
        # Ejemplo: si p_c = 0.5778, el umbral es el percentil 42.22%
        percentil_corte = (1.0 - p_c) * 100.0
        char_th_bbse[c] = np.percentile(scores_test[:, c], percentil_corte)

# Predicciones finales con los umbrales derivados de BBSE
preds_test_bbse = (scores_test >= char_th_bbse).astype(int)

In [281]:
print(compute_metrics(y_test, preds_test))

{'accuracy': 0.06161719549641761, 'f1': 0.1703028235695136, 'precision': 0.1317639812359233, 'recall': 0.4640139605686214, 'hamming_loss': 0.07346103231466589}
